Connect to the GOOGLE CLOUD

In [ ]:
# from google.colab import files
# files.upload()

Install Dependancies

In [ ]:
!pip install "torch>=1.3" "datasets==2.0.0" "sentencepiece!=0.1.92" "transformers==4.25.0" "nltk==3.6.5" protobuf tensorboardX wandb jieba nltk evaluate

Load Utilities

In [ ]:
import glob
import logging
import os
import random
import re
from typing import *

import numpy as np
import torch
from tqdm.auto import tqdm


#TODO fix logging in this file


def get_save_dir(base_dir, name):
    """
    Args:
        base_dir (str): Base directory in which to make save directories.
        name (str): Name to identify this training run. Need not be unique.

    Returns:
        save_dir (str): Path to a new directory with a unique name.
    """
    base_path = os.path.join(base_dir, name)
    match_dirs = sorted(glob.glob(base_path + "*"))

    if len(match_dirs) == 0:  # none existed
        unused_dir = base_path + "-01"
    else:       # increment from the last one
        last = match_dirs[-1]
        r = re.match(".*-(\d\d)", last)
        next_num = int(r.group(1)) + 1

        unused_dir = base_path + f'-{next_num:02d}'

    # log.info(f'Will create new save_dir: {unused_dir}')
    os.makedirs(unused_dir)
    return unused_dir


def visualize(tbx, pred_dict: Union[Dict, List], step, split, num_visuals):
    """Visualize text examples to TensorBoard.

    Args:
        tbx (tensorboardX.SummaryWriter): Summary writer.
    """
    if num_visuals <= 0:
        return
    if num_visuals > len(pred_dict):
        num_visuals = len(pred_dict)

    for i in range(num_visuals):
        # unpack tuple
        orig_input, orig_target, actual_output = pred_dict[i]

        tbl_fmt = (f'- **Source:** {orig_input}\n'
                   + f'- **Target:** {orig_target}\n'
                   + f'- **Predicted:** {actual_output}\n')
        tbx.add_text(tag=f'{split}/{i+1}_of_{num_visuals}',
                     text_string=tbl_fmt,
                     global_step=step)


def save_preds(preds: List[Tuple[str,str,str]], save_dir, file_name='predictions.csv'):
    """Save predictions `preds` to a CSV file named `file_name` in `save_dir`.

    Args:
        preds (list): List of predictions each of the form (source, target, actual),
        save_dir (str): Directory in which to save the predictions file.
        file_name (str): File name for the CSV file.

    Returns:
        save_path (str): Path where CSV file was saved.
    """
    save_path = os.path.join(save_dir, file_name)
    np.savetxt(save_path, np.array(preds), delimiter='@', fmt='%s')

    return save_path

# This function is like a rouge metric
def masked_token_match(tgt_ids: torch.tensor, outputs: torch.tensor,
                       return_indices=False) -> Union[Tuple[int,int], Tuple[int, int, torch.tensor]]:
    """
    Takes generated outputs and tgt_ids, both of size (batch_size, seq_len), where seq_len may differ
    For all tokens in tgt_ids that are not PAD or EOS,
        - check that they are equal
        - count all the examples that are an exact match

    Returns:
        - total_matches_no_eos: all the matches where we get everything except EOS correct
        - total_matches_with_eos: all matches where we get everything including EOS
        - optional (if return_indices): the indices where we have a match on everything up to the EOS token

    """
    # left-shift
    # assert (output_ids[:,0] == 0)       # T5 should start with a pad token; other models could vary
    output_shifted = outputs[:,1:]

    if output_shifted.shape <= tgt_ids.shape:
        # create output_padded, which truncates output at tgt_ids size, filling with pad tokens
        output_padded = torch.zeros_like(tgt_ids)
        output_padded[:output_shifted.shape[0], :output_shifted.shape[1]] = output_shifted
    else:       # output_shifted is bigger
        # so copy only up to the target IDs length
        output_padded = output_shifted[:,:tgt_ids.shape[1]]     # copy all rows (bs) and up to tgt_ids length

    # compare where tokens are > 1 (i.e. not pad or EOS)
    match_indices = output_padded == tgt_ids          # either they match
    matches_no_eos = torch.logical_or(match_indices, tgt_ids < 2)   # or we ignore them (pad and eos)
    matches_with_eos = torch.logical_or(match_indices, tgt_ids < 1) # or we ignore them (just pad)
    total_matches_no_eos = torch.sum(torch.all(matches_no_eos, axis=1))
    total_matches_with_eos = torch.sum(torch.all(matches_with_eos, axis=1))

    correct_indices = torch.nonzero(torch.all(matches_no_eos, axis=1))

    if return_indices:
        return total_matches_no_eos, total_matches_with_eos, correct_indices
    else:
        return total_matches_no_eos, total_matches_with_eos


# We use this for evaluation
class AverageMeter:
    """Keep track of average values over time.

    Adapted from:
        > https://github.com/pytorch/examples/blob/master/imagenet/main.py
    """
    def __init__(self):
        self.avg = 0
        self.sum = 0
        self.count = 0

    def reset(self):
        """Reset meter."""
        self.__init__()

    def update(self, val, num_samples=1):
        """Update meter with new value `val`, the average of `num` samples.

        Args:
            val (float): Average value to update the meter with.
            num_samples (int): Number of samples that were averaged to
                produce `val`.
        """
        self.count += num_samples
        self.sum += val * num_samples
        self.avg = self.sum / self.count

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_available_devices():
    """Get IDs of all available GPUs.

    Returns:
        device (torch.device): Main device (GPU 0 or CPU).
        gpu_ids (list): List of IDs of all GPUs that are available.
    """
    gpu_ids = []
    if torch.cuda.is_available():
        gpu_ids += [gpu_id for gpu_id in range(torch.cuda.device_count())]
        device = torch.device(f'cuda:{gpu_ids[0]}')
        torch.cuda.set_device(device)
    else:
        device = torch.device('cpu')

    return device, gpu_ids

def get_logger(log_dir, name, log_level="debug"):
    """Get a `logging.Logger` instance that prints to the console
    and an auxiliary file.

    Args:
        log_dir (str): Directory in which to create the log file.
        name (str): Name to identify the logs.

    Returns:
        logger (logging.Logger): Logger instance for logging events.
    """
    class StreamHandlerWithTQDM(logging.Handler):
        """Let `logging` print without breaking `tqdm` progress bars.

        See Also:
            > https://stackoverflow.com/questions/38543506
        """
        def emit(self, record):
            try:
                msg = self.format(record)
                tqdm.write(msg)
                self.flush()
            except (KeyboardInterrupt, SystemExit):
                raise
            except:
                self.handleError(record)

    # Create logger
    logger = logging.getLogger(name)
    if log_level == "debug":
        logger.setLevel(logging.DEBUG)
    elif log_level == "info":
        logger.setLevel(logging.INFO)
    else:
        raise ValueError(f"Invalid log level {log_level}")

    # Log everything (i.e., DEBUG level and above) to a file
    log_path = os.path.join(log_dir, 'log.txt')
    file_handler = logging.FileHandler(log_path)
    file_handler.setLevel(logging.DEBUG)

    # Log everything except DEBUG level (i.e., INFO level and above) to console
    console_handler = StreamHandlerWithTQDM()
    console_handler.setLevel(logging.INFO)

    # Create format for the logs
    file_formatter = logging.Formatter('[%(asctime)s] %(message)s',
                                       datefmt='%m.%d.%y %H:%M:%S')
    file_handler.setFormatter(file_formatter)
    # console_formatter = logging.Formatter('[%(asctime)s] %(message)s',
    #                                       datefmt='%m.%d.%y %H:%M:%S')
    console_formatter = logging.Formatter(
        '[%(asctime)s] [%(filename)s:%(lineno)s - %(funcName)s()] %(message)s',
        datefmt='%m.%d %H:%M:%S')
    console_handler.setFormatter(console_formatter)

    # add the handlers to the logger
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger

def generate_seeded_dataset(input_file, output_file, ratio):
    with open(input_file, 'r') as f_target, open(output_file, 'w') as f_source:
        for target_line in f_target:
            source_line = target_line.strip().split()
            source_line = " ".join(source_line[:round(len(source_line) * ratio)])
            f_source.write('%s\n' % source_line)

Load Models and Tokenizers

In [ ]:
import os
import argparse
import traceback
from pathlib import Path

import socket
from collections import OrderedDict
from typing import *

import torch
import torch.nn as nn
import wandb
from tensorboardX import SummaryWriter
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import AdamW, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    MBartForConditionalGeneration,
    MBartTokenizer,
    MBart50Tokenizer,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    AutoModelForSeq2SeqLM, # for indicBART
    AlbertTokenizer, #https://huggingface.co/ai4bharat/IndicBART
    AutoTokenizer,
)

# seed for random
seed = 42

lang_dict = {
    "en" : "English",
    "si" : "Sinhala",
    "ta" : "Tamil",
}

model_dict = {
    "mt5-base" : "google/mt5-base",
    "mbart-large-50" : "facebook/mbart-large-50",
    "m2m100_418M": "facebook/m2m100_418M",
    "indic-bartSS": "ai4bharat/IndicBARTSS",
}

tokenizer_lang_dic = {
    "en" : "en_XX",
    "si" : "si_LK",
    "ta" : "ta_IN",
    "dv" : "si_LK"
}

def get_model(model_name, weight_path = ""):

    if weight_path =="":
        model_path = model_dict[model_name]
    else:
        model_path = weight_path

    if(model_name == "mt5-base"):
        model = MT5ForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "mbart-large-50"):
        model = MBartForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "m2m100_418M"):
        model = M2M100ForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "indic-bartSS"):
        model = MBartForConditionalGeneration.from_pretrained(model_path)
        # Or use model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

    return model

def get_tokenizer(model_name, language = 'en', weight_path = ""):

    language_label = tokenizer_lang_dic[language]

    if weight_path =="":
        model_path = model_dict[model_name]
    else:
        model_path = weight_path

    if(model_name == "indic-bartSS"):
        tokenizer = AutoTokenizer.from_pretrained(model_path, do_lower_case=False, use_fast=False, keep_accents=True)
        # Or use tokenizer = AlbertTokenizer.from_pretrained(model_path, do_lower_case=False, use_fast=False, keep_accents=True)

    elif(model_name == "mt5-base"):
        tokenizer = MT5Tokenizer.from_pretrained(model_path)

    elif(model_name == "mbart-large-50"):
        tokenizer = MBart50Tokenizer.from_pretrained(model_path, src_lang=language_label, tgt_lang=language_label)

    elif(model_name == "m2m100_418M"):
        tokenizer = M2M100Tokenizer.from_pretrained(model_path)
        tokenizer.src_lang = language
        tokenizer.tgt_lang = language

    return tokenizer

Load DataLoader Functions

In [ ]:

# A dataset for our inputs.
class MWPDataSet(Dataset):
    def __init__(self, tokenizer, hf_dataset, max_src_len=200, max_tgt_len=500):
        """
        max_examples: if > 0 then will load only max_examples into the dataset; -1 means use all

        max_src and max_tgt len refer to number of tokens in the input sequences
        # Note: these are not randomized. If they were we might need to collate.
        """

        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.input_text = hf_dataset['source']
        self.target_text = hf_dataset['target']

        self.inputs = []
        self.targets = []

        self._build()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        source_ids = self.inputs[index]["input_ids"].squeeze()
        target_ids = self.targets[index]["input_ids"].squeeze()

        src_mask    = self.inputs[index]["attention_mask"].squeeze()  # might need to squeeze
        target_mask = self.targets[index]["attention_mask"].squeeze()  # might need to squeeze

        src_text = self.input_text[index]
        tgt_text = self.target_text[index]

        # These will be cast to torch.long in forward
        return {"source_ids": source_ids, "source_mask": src_mask,
                "target_ids": target_ids, "target_mask": target_mask,
                "source_text": src_text, "target_text": tgt_text}

    def _build(self):
        for src, tgt in zip(self.input_text, self.target_text):
            tokenized_inputs = self.tokenizer(
                src, max_length=self.max_src_len, padding="max_length", return_tensors="pt", truncation=True
            )
            tokenized_targets = self.tokenizer(
                tgt, max_length=self.max_tgt_len, padding="max_length", return_tensors="pt", truncation=True
            )
            self.inputs.append(tokenized_inputs)
            self.targets.append(tokenized_targets)

"""
Returns: Tuple[train_loader : DataLoader, dev_loader : DataLoader]
# Note:
# - we default to not shuffling the dev set
"""
def get_dataloaders(tokenizer, batch_size, train_data, val_data, test_data, num_workers, k_max_src_len, k_max_tgt_len,
                    shuffle_train=True, shuffle_dev=False,shuffle_test=False) -> Tuple[DataLoader, DataLoader, DataLoader]:

    #TODO should pass max src and max tgt len in as arguments
    train_data_set = MWPDataSet(tokenizer, train_data, max_src_len=k_max_src_len, max_tgt_len=k_max_tgt_len)
    eval_data_set = MWPDataSet(tokenizer, val_data, max_src_len=k_max_src_len, max_tgt_len=k_max_tgt_len)
    test_data_set = MWPDataSet(tokenizer, test_data, max_src_len=k_max_src_len, max_tgt_len=k_max_tgt_len)

    train_loader = DataLoader(train_data_set, batch_size=batch_size, shuffle=shuffle_train, num_workers=num_workers)
    eval_loader = DataLoader(eval_data_set, batch_size=batch_size, shuffle=shuffle_dev, num_workers=num_workers)
    test_loader = DataLoader(test_data_set, batch_size=batch_size, shuffle=shuffle_test, num_workers=num_workers)
    log.info(f'Datasets loaded with sizes: train: {len(train_data_set)}, dev: {len(eval_data_set)}, test: {len(test_data_set)}')

    return train_loader, eval_loader , test_loader

def forward(model, device, batch):
    src_ids = batch["source_ids"].to(device, dtype=torch.long)
    src_mask = batch["source_mask"].to(device, dtype=torch.long)
    tgt_ids = batch["target_ids"].to(device, dtype=torch.long)

    # padded ids (pad=0) are set to -100, which means ignore for loss calculation
    tgt_ids[tgt_ids[: ,:] == 0 ] = -100
    label_ids = tgt_ids.to(device)
    # when we call model() with labels, they will be
    # - automatically right shifted by 1 (for teacher forcing)
    # - prepended by BOS=Beginning of sequence which is a PAD token
    # - any token that was -100 will be masked_fill_ to <pad> for teacher forcing
    # return_dict means return as a dictionary
    out_dict = model(src_ids, attention_mask=src_mask, labels=label_ids, return_dict=True)
    loss, logits = out_dict['loss'], out_dict['logits']
    return loss, logits

def write_output_to_text(pred_list_all,save_path):
  with open(save_path, 'w') as f:
    for item in pred_list_all:
        f.write("{0}\n".format(item))

Model Training and Evaluation Loop

In [ ]:
def main(model_name, lang, experiment_id, experiment_name, use_wandb, seed, epochs, lr, adam_eps,
         scheduler, warmup_steps, max_grad_norm,batch_size, num_workers,max_src_len,max_tgt_len,
         max_length, use_trained, weight_dir, get_testing_results, train_dataset,
         val_dataset, test_dataset, enable_early_stopping = True, enable_checkpoints = False, early_stopping_patience = 5,
         early_stopping_threshold = 0.001):

    save_dir = f'./save/{experiment_name}'
    save_weights_dir = f'./save_weights/{experiment_name}'
    log_dir = f'./logs/{experiment_name}'

    set_seed(seed)
    language = lang_dict[lang]
    comment=\
        """
        Started finetuning {} for MWP Generation...
        Using mulipleSentenceDataset.
        """.format(model_name)

    print("=============================================================")
    print("DEBUG: experiment_name = {}".format(experiment_name))
    print("DEBUG: save_dir = {}".format(save_dir))
    print("DEBUG: save_weights_dir = {}".format(save_weights_dir))
    print("DEBUG: log_dir = {}".format(log_dir))

    Path(save_dir).mkdir(parents=True, exist_ok=True)
    Path(save_weights_dir).mkdir(parents=True, exist_ok=True)
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    # Save the predicted text file paths
    save_predicted_eval_output_text_dir = os.path.join(log_dir, 'eval.txt')
    save_predicted_test_output_text_dir =  os.path.join(log_dir, 'test.txt')

    if use_wandb:
        wandb.init()
        record_dir = wandb.run.dir
    else:
        record_dir = get_save_dir(save_dir, experiment_name)

    global log
    log = get_logger(record_dir, "root", "debug")
    tbx = SummaryWriter(record_dir, flush_secs=5)
    log.info(experiment_name)
    log.info(comment)


    all_config = {
        "save_dir": save_dir,
        "epochs": epochs,
        "model": model_name,
        "lr": lr,
        "adam_eps": adam_eps,
        "warmup": warmup_steps,
        "workers": num_workers,
        "max grad": max_grad_norm,
        "batch_size": batch_size,
        "max_src_len": max_src_len,
        "max_tgt_len": max_tgt_len
    }

    #Initialize model and tokenizer
    device, gpu_ids = get_available_devices()


    if(use_trained):
        weight_path = weight_dir
        model = get_model(model_name, weight_path)
        model.config.max_length = max_length
        tokenizer = get_tokenizer(model_name, lang, weight_path)
    else:
        model = get_model(model_name)
        model.config.max_length = max_length
        tokenizer = get_tokenizer(model_name, lang)

    # print(model.config)
    print("----------------------------")
    print(model.config)
    print("----------------------------")

    train_loader, dev_loader, test_loader = \
        get_dataloaders(tokenizer,batch_size=batch_size,train_data=train_dataset,val_data=val_dataset,test_data=test_dataset,num_workers=num_workers,k_max_src_len=max_src_len,
                        k_max_tgt_len=max_tgt_len)

    # reset in case we used the -1 flag for all
    num_train = len(train_loader.dataset)
    num_val = len(dev_loader.dataset)
    num_test = len(test_loader.dataset)

    total_steps = ( (num_train // batch_size) * epochs)     # num times that optim.step() will be called
    total_train = num_train * epochs

    model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr, eps=adam_eps)

    if(scheduler == "linear"):
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                                    num_training_steps=total_steps)
    else:
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                                    num_training_steps=total_steps)

    log.info(f'device: {device}\n'
             f'gpu_ids: {gpu_ids}\n'
             f'total_steps: {total_steps}\n'
             f'total_train (num_t * epoch): {total_train}\n'
             f'machine: {socket.gethostname()}\n')

    config_str = "\n"
    for k, v in all_config.items():
        config_str += f'{k}: {v}\n'
    config_str += f'record_dir: {record_dir}\n'
    log.info(config_str)

    # --- Early Stopping Initialization ---
    best_eval_loss = float('inf')
    epochs_without_improvement = 0
    training_should_continue = True

    # --- Training Loop ---
    epoch = 0       # number of times we have passed through entire set of training examples
    step = 0        # number of total examples we have done (will be epoch * len(data_set) at end of each epoch)
    while epoch < epochs and training_should_continue:
        epoch += 1
        model.train()
        log.info(f'Training at epoch {epoch}... ')
        with torch.enable_grad(), tqdm(total=num_train) as progress_bar:
            for batch_num, batch in enumerate(train_loader):
                batch_size = len(batch["source_ids"])
                loss, logits = forward(model, device, batch)
                loss_val = loss.item()      # get the item since loss is a tensor

                # Backward
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                scheduler.step()        # don't need to pass step to scheduler

                # Log info
                step += batch_size
                progress_bar.update(batch_size)
                progress_bar.set_postfix(epoch=epoch,
                                        loss=loss_val)
                tbx.add_scalar('train/loss', loss_val, step)
                tbx.add_scalar('train/LR',
                              optimizer.param_groups[0]['lr'],
                              step)
        # --- Save checkpoint every 5 epochs if early stopping is disabled ---
        if not enable_early_stopping and enable_checkpoints and epoch % 5 == 0:
          checkpoint_epoch_dir = os.path.join(save_weights_dir, f'checkpoint_epoch_{epoch}')
          Path(checkpoint_epoch_dir).mkdir(parents=True, exist_ok=True)
          log.info(f'Saving checkpoint at epoch {epoch} to {checkpoint_epoch_dir}...')
          model.save_pretrained(checkpoint_epoch_dir)
          tokenizer.save_pretrained(checkpoint_epoch_dir)

        ###############
          # Periodic Evaluation for Early Stopping when enabled
        ###############
        if(enable_early_stopping==True):
          log.info(f'Evaluating after epoch {epoch}...')
          model.eval()        # put model in eval mode

          loss_meter = AverageMeter()    # NLL (default metric for model) (reset each time)

          # set up two count variables
          total_matches_no_eos_ct = 0
          total_matches_with_eos_ct = 0

          with torch.no_grad(), \
                tqdm(total=num_val, desc=f"Epoch {epoch} Evaluation") as progress_bar:
                for batch_num, batch in enumerate(dev_loader):
                    batch_size = len(batch["source_ids"])

                    # evaluation for loss fcn
                    loss, _ = forward(model, device, batch)     # loss, logits, but don't need logits
                    loss_meter.update(loss.item(), batch_size)  # loss.item() since it's a tensor

                    progress_bar.update(batch_size)
                    progress_bar.set_postfix(eval_loss=loss_meter.avg)

          current_eval_loss = loss_meter.avg
          log.info(f"Epoch {epoch} Evaluation Loss (NLL): {current_eval_loss:.4f}")
          if tbx: tbx.add_scalar('eval/epoch_loss', current_eval_loss, epoch)

          # --- Early Stopping Logic ---
          if current_eval_loss < best_eval_loss - early_stopping_threshold:
              log.info(f"Validation loss improved from {best_eval_loss:.4f} to {current_eval_loss:.4f}.")
              best_eval_loss = current_eval_loss
              epochs_without_improvement = 0
              log.info(f"Saving best model at epoch {epoch} to {save_weights_dir}...")

              model.save_pretrained(save_weights_dir)
              tokenizer.save_pretrained(save_weights_dir)
          else:
              epochs_without_improvement += 1
              log.info(f"Validation loss did not improve significantly ({current_eval_loss:.4f} vs best {best_eval_loss:.4f}). Patience: {epochs_without_improvement}/{early_stopping_patience}.")

          if epochs_without_improvement >= early_stopping_patience:
              log.info(f"Early stopping triggered after {early_stopping_patience} epochs without improvement.")
              training_should_continue = False # This will break the outer while loop

    log.info("Training loop finished or early stopping triggered.")
    ###############
    # TEST (you might want to save checkpoints)
    ###############
    if epoch < epochs and enable_early_stopping:
        log.info(f"Loading best model from {save_weights_dir} for final operations...")
        model = get_model(model_name, save_weights_dir)
        tokenizer = get_tokenizer(model_name, lang, save_weights_dir)
        model.to(device) # Ensure it's on the correct device
        log.info("Best model loaded.")

    if(get_testing_results==True):
        log.info(f'Testing at step {step}...')
        model.eval()        # put model in eval mode

        # See how the model is doing with exact match on tokens
        test_pred_list_all = []                      # accumulate for saving; list; one list per epoch
        test_pred_list_correct = []
        loss_meter = AverageMeter()    # NLL (default metric for model) (reset each time)

        Test_prediction_list_for_write_to_text = []

        # set up two count variables
        total_matches_no_eos_ct = 0
        total_matches_with_eos_ct = 0

        with torch.no_grad(), \
            tqdm(total=num_test) as progress_bar:
            for batch_num, batch in enumerate(test_loader):
                batch_size = len(batch["source_ids"])

                # testing for loss fcn
                loss, _ = forward(model, device, batch)     # loss, logits, but don't need logits
                loss_meter.update(loss.item(), batch_size)  # loss.item() since it's a tensor

                # predict / generate for token matches
                src_ids = batch["source_ids"].to(device, dtype=torch.long)
                src_mask = batch["source_mask"].to(device, dtype=torch.long)
                tgt_ids = batch["target_ids"].to(device, dtype=torch.long)
                # note you could tweak the generation params. See huggingface details for generate
                generated_ids = model.generate(src_ids, attention_mask=src_mask, max_length=None, min_length=None)       # (batch x seq length) -------Added min length

                # collect some stats
                total_matches_no_eos, total_matches_with_eos, correct_indices = \
                      masked_token_match(tgt_ids, generated_ids, return_indices=True)
                total_matches_no_eos_ct += total_matches_no_eos
                total_matches_with_eos_ct += total_matches_with_eos

                # save for qualitative analysis
                orig_text_input, orig_text_output = batch["source_text"], batch["target_text"]
                #TODO this could break once skip_special_tokens is fixed
                outputs_decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)  #skip_special_tokens=False
                preds = list(zip(orig_text_input, orig_text_output, outputs_decoded))
                test_pred_list_all.extend(preds)

                # ----------print-----------
                print(outputs_decoded)
                Test_prediction_list_for_write_to_text.extend(outputs_decoded)

                # we also store only the correct indices
                for idx in correct_indices.tolist():    # tensor to list; these are the valid indices
                    test_pred_list_correct.append(preds[idx[0]])     # each item was a list of one element

                # print one batch of generations for qualitative assessment
                if batch_num == 0:
                    for orig_input, orig_target, actual_output in preds[:1]:
                        log.info(f'Source: {orig_input}\t Target: {orig_target}\n'
                                f'\t Actual: {actual_output}')

                 # Log info
                progress_bar.update(batch_size)
                progress_bar.set_postfix(NLL=loss_meter.avg)

          # save predictions for qualititative analysis
        save_preds(test_pred_list_all, record_dir, file_name="preds_all_test.csv")
        save_preds(test_pred_list_correct, record_dir, file_name="preds_correct_test.csv")
        results_list = [('NLL', loss_meter.avg),
                        ('exact_match_with_eos', total_matches_with_eos_ct),
                        ('exact_match_no_eos', total_matches_no_eos_ct)]
        results = OrderedDict(results_list)


        # Save predictions to a output file
        write_output_to_text(Test_prediction_list_for_write_to_text,save_predicted_test_output_text_dir)


        # Log to console
        results_str = ', '.join(f'{k}: {v:05.2f}' for k, v in results.items())
        log.info(f'test {results_str}')
    if(enable_early_stopping==False):
      final_save_path = os.path.join(save_weights_dir, 'final_model')
      Path(final_save_path).mkdir(parents=True, exist_ok=True)
      model.save_pretrained(final_save_path)
      tokenizer.save_pretrained(final_save_path)
      log.info(f"Final model saved to {final_save_path}")
    log.info("Script finished.")


Training Procedure of mBART Model

In [ ]:
# Model names
# "mt5-base" : "google/mt5-base",
# "mbart-large-50" : "facebook/mbart-large-50",
# "m2m100_418M": "facebook/m2m100_418M",
# "indic-bartSS": "ai4bharat/IndicBARTSS",

model = 'mbart-large-50'
lang = # si or ta
experiment_name = # Experiment Name

use_wandb = False

# Hyperparameters
seed = 42
# You need to change epochs if needed
epochs = 20
lr = 1e-4
adam_eps = 1e-8
scheduler = 'linear'
warmup_steps = 0
max_grad_norm = 1.0
batch_size = 4
num_workers = 4
max_src_len = 200
max_tgt_len = 400
max_length = 200

# True if you want to Load prev checkpoints
use_trained = False
weight_dir = './save_weights/tamil_J_mBART/final_model'
# enable test stage -> generate mwps for test dataset and save them
get_testing_results = False
# enable early stopping based on eval set
enable_early_stopping = False
# enable saving checkpoint by each 5 epochs
enable_checkpoints = False

Load Dataset

In [ ]:
# Load test, eval and train data and split it to source and targets

# Create Dataset objects for each split (train, dev, test)
# train_dataset = Dataset.from_dict({'source': train_sources, 'target': train_targets})
# val_dataset = Dataset.from_dict({'source': dev_sources, 'target': dev_targets})
# test_dataset = Dataset.from_dict({'source': test_sources, 'target': test_targets})

Start the Training Loop

In [ ]:
# If calling script then execute
if __name__ == "__main__":
    try:
        main(model, lang, experiment_id, experiment_name, use_wandb, seed, epochs, lr, adam_eps,
             scheduler, warmup_steps, max_grad_norm,batch_size, num_workers,max_src_len,max_tgt_len,
             max_length, use_trained, weight_dir, get_testing_results, train_dataset, val_dataset, test_dataset, enable_early_stopping, enable_checkpoints)
    except Exception as e:
        print("Error: {0}".format(e))
        print(traceback.format_exc())